# LangGraph vs CrewAI vs DSPy: End-to-End Notebook

Companion notebook to `article.md`. Runs the same bench, probes, and compile
experiment the article reports, against three Gemini Flash variants.

**Quick start**

1. `cp .env.example .env` and add your `GEMINI_API_KEY`.
2. `python3.13 -m venv .venv && source .venv/bin/activate && pip install -r requirements.txt`
3. Open this notebook and Run All. Set `RUN_LIVE = False` below to skip live API calls and only display saved results.

The full live re-run costs roughly $5-8 in Gemini API spend and takes about 90 minutes serial. The cached path (RUN_LIVE = False) uses results from `results/*.json` and runs in seconds.


## 1. Setup

In [1]:
import os, sys, json, time
from pathlib import Path

ROOT = Path('.').resolve()
sys.path.insert(0, str(ROOT / 'bench'))

# Load .env
env_path = ROOT / '.env'
if env_path.exists():
    for line in env_path.read_text().splitlines():
        if '=' in line and not line.startswith('#'):
            k, v = line.split('=', 1)
            os.environ.setdefault(k.strip(), v.strip())

RUN_LIVE = bool(os.environ.get('GEMINI_API_KEY')) and os.environ.get('RUN_LIVE', '0') == '1'
print('GEMINI_API_KEY set:', bool(os.environ.get('GEMINI_API_KEY')))
print('RUN_LIVE:', RUN_LIVE)


GEMINI_API_KEY set: True
RUN_LIVE: False


## 2. Dataset

100 unique refund tickets covering 10 issue types (damaged, defective, wrong-item, late, missing-parts, incompatible, quality, change-of-mind, warranty, sizing), 4 length buckets, 5 tones, 5 amount buckets, 3 policy-fit categories. Generator is deterministic (seed=42).


In [2]:
# Show the generator (or regenerate if needed)
ds_path = ROOT / 'bench' / 'tickets_100.json'
if not ds_path.exists():
    !python bench/gen_tickets.py
tickets = json.loads(ds_path.read_text())
print(f'tickets: {len(tickets)}')
print('sample:', tickets[0])


tickets: 100
sample: {'ticket_id': 'T-1001', 'email_text': "Hope you're well. toaster stopped working after 2 days. Appreciate your help.", 'requested_refund': 27.67, 'meta': {'issue': 'defective', 'policy_fit': 'damage_yes', 'category': 'kitchen', 'product': 'toaster', 'length_bucket': 'short', 'tone': 'calm', 'days_since_delivery': 7, 'amount_bucket': '20-75', 'email_char_len': 77, 'email_word_len': 12}}


## 3. The three implementations

**LangGraph** (state-machine first; checkpoint at every superstep):

In [3]:
(ROOT / 'bench' / 'impl_langgraph.py').read_text()

'"""LangGraph refund assistant. State-first, checkpointed, interruptible."""\nfrom __future__ import annotations\n\nimport os\nfrom typing import TypedDict\n\nfrom langchain_google_genai import ChatGoogleGenerativeAI\nfrom langgraph.checkpoint.memory import InMemorySaver\nfrom langgraph.graph import END, START, StateGraph\nfrom langgraph.types import Command, interrupt\n\n\nclass RefundState(TypedDict, total=False):\n    ticket_id: str\n    email_text: str\n    policy_summary: str\n    requested_refund: float\n    triage: str\n    draft_reply: str\n    approved: bool\n    sent: bool\n\n\ndef _llm():\n    return ChatGoogleGenerativeAI(\n        model=os.environ.get("GEMINI_MODEL", "gemini-2.5-flash"),\n        google_api_key=os.environ["GEMINI_API_KEY"],\n        temperature=0,\n    )\n\n\ndef _text(msg) -> str:\n    """Normalize Gemini response content: Gemini 2.x returns str, Gemini 3 returns list[dict]."""\n    c = msg.content\n    if isinstance(c, str):\n        return c\n    if isi

**CrewAI** (roles inside a Flow; OSS HITL only):

In [4]:
(ROOT / 'bench' / 'impl_crewai.py').read_text()

'"""CrewAI refund assistant. Role-first, OSS-compatible (no @human_feedback enterprise decorator)."""\nfrom __future__ import annotations\n\nimport os\n\nfrom crewai import Agent, Crew, LLM, Process, Task\nfrom crewai.flow.flow import Flow, listen, start\nfrom pydantic import BaseModel\n\n\nclass RefundState(BaseModel):\n    ticket_id: str = ""\n    email_text: str = ""\n    policy_summary: str = "Damaged items refundable within 30 days."\n    requested_refund: float = 0.0\n    triage: str = ""\n    draft_reply: str = ""\n    approved: bool | None = None\n    status: str = ""\n\n\ndef _llm():\n    return LLM(\n        model=f"gemini/{os.environ.get(\'GEMINI_MODEL\', \'gemini-2.5-flash\')}",\n        api_key=os.environ["GEMINI_API_KEY"],\n        temperature=0,\n    )\n\n\ndef build_agents():\n    llm = _llm()\n    policy = Agent(\n        role="Refund Policy Analyst",\n        goal="Decide eligibility from ticket and policy",\n        backstory="You know damage clauses and return windo

**DSPy** (Signature + ChainOfThought; cache disabled for fairness):

In [5]:
(ROOT / 'bench' / 'impl_dspy.py').read_text()

'"""DSPy refund assistant. Program-first, compile-ready."""\nfrom __future__ import annotations\n\nimport os\n\nimport dspy\n\n\ndef configure():\n    dspy.configure(lm=dspy.LM(\n        f"gemini/{os.environ.get(\'GEMINI_MODEL\', \'gemini-2.5-flash\')}",\n        api_key=os.environ["GEMINI_API_KEY"],\n        temperature=0,\n        cache=False,\n    ))\n\n\nclass Triage(dspy.Signature):\n    """Decide refund eligibility given ticket and policy."""\n    email_text: str = dspy.InputField()\n    policy: str = dspy.InputField()\n    eligible: bool = dspy.OutputField()\n    rationale: str = dspy.OutputField()\n\n\nclass Reply(dspy.Signature):\n    """Draft a concise customer support reply (3 sentences max, no sign-off)."""\n    email_text: str = dspy.InputField()\n    rationale: str = dspy.InputField()\n    reply: str = dspy.OutputField()\n\n\nclass RefundProgram(dspy.Module):\n    def __init__(self):\n        super().__init__()\n        self.triage = dspy.ChainOfThought(Triage)\n        s

## 4. Run one ticket through each framework

This calls the live Gemini API once per framework if `RUN_LIVE`. Otherwise skipped.


In [6]:
if RUN_LIVE:
    os.environ['GEMINI_MODEL'] = 'gemini-2.5-flash-lite'
    sample = tickets[0]
    for name, mod in [('langgraph', 'impl_langgraph'), ('crewai', 'impl_crewai'), ('dspy', 'impl_dspy')]:
        m = __import__(mod)
        t0 = time.perf_counter()
        out = m.run(sample)
        dt = time.perf_counter() - t0
        print(f'{name}: {dt:.2f}s  draft_reply: {(out.get("draft_reply") or out.get("reply") or "")[:120]!r}')
else:
    print('RUN_LIVE=False; skipping single-ticket demo.')


RUN_LIVE=False; skipping single-ticket demo.


## 5. The 100-ticket bench

The full bench runs 100 tickets across all three frameworks per model, captures latency, prompt/completion tokens, and LLM call count. Live runtime is ~10-12 minutes per model.


In [7]:
if RUN_LIVE:
    # Run all three models
    !cd bench && python bench.py --model gemini-2.5-flash
    !cd bench && python bench.py --model gemini-2.5-flash-lite
    !cd bench && python bench.py --model gemini-3.1-flash-lite-preview


**Cached results: latency table with 95% CIs (computed by `enrich.py`).**

In [8]:
!python bench/enrich.py 2>/dev/null | head -30

Wrote /Users/praneeth.paikray/Documents/Code/articles/langgraph_vs_crewai/results/analysis_v2.md
# Benchmark Analysis (v2, audited)

## Latency (mean with 95% CI; bootstrap p95/p99 with 95% CI)

| Model | Framework | n | Mean | 95% CI | Median | p95 | p95 95% CI | p99 | p99 95% CI |
|---|---|---|---|---|---|---|---|---|---|
| gemini-2.5-flash-lite | langgraph | 100 | 2.12 | [2.05, 2.18] | 2.05 | 2.61 | [2.44, 3.07] | 3.17 | [2.61, 4.23] |
| gemini-2.5-flash-lite | crewai | 100 | 2.71 | [2.61, 2.81] | 2.60 | 3.59 | [3.26, 3.84] | 4.14 | [3.59, 5.50] |
| gemini-2.5-flash-lite | dspy | 100 | 1.83 | [1.67, 1.99] | 1.54 | 3.44 | [3.02, 3.88] | 3.88 | [3.44, 4.19] |
| gemini-2.5-flash | langgraph | 100 | 6.99 | [6.37, 7.60] | 5.80 | 14.03 | [10.16, 17.23] | 19.11 | [14.03, 21.82] |
| gemini-2.5-flash | crewai | 100 | 7.28 | [6.13, 8.44] | 5.76 | 14.50 | [8.77, 28.56] | 41.24 | [14.50, 43.08] |
| gemini-2.5-flash | dspy | 100 | 4.37 | [3.98, 4.77] | 3.39 | 8.49 | [7.52, 9.11] | 10.68 | [8.49,

## 6. Probes

Four probes pull additional structure out of the bench:

- `probe_overhead.py` runs a single fixed prompt through each framework to isolate template overhead.
- `inspect_messages.py` captures the literal byte-level prompt each framework sends.
- `probe_reasoning.py` decomposes completion tokens into reasoning vs non-reasoning per framework on Gemini 2.5 Flash.
- `probe_cprofile.py` wraps a single-ticket run in cProfile per framework and captures function call counts.


In [9]:
if RUN_LIVE:
    !python bench/probe_overhead.py
    !python bench/inspect_messages.py
    !python bench/probe_reasoning.py
    !python bench/probe_cprofile.py


**Cached: framework prompt overhead (probe_overhead.json):**

In [10]:
p = ROOT / 'results' / 'probe_overhead.json'
if p.exists():
    d = json.loads(p.read_text())
    for model, rows in d.items():
        print(f'\n=== {model} ===')
        for r in rows:
            print(f'  {r}')



=== gemini-2.5-flash ===
  {'framework': 'raw', 'latency_s': 3.042, 'prompt_tokens': 26, 'completion_tokens': 279, 'total_tokens': 305}
  {'framework': 'langgraph', 'latency_s': 2.89, 'prompt_tokens': 26, 'completion_tokens': 279, 'total_tokens': 305}
  {'framework': 'crewai', 'latency_s': 1.998, 'prompt_tokens': 182, 'completion_tokens': 118, 'total_tokens': 300, 'llm_calls': 1}
  {'framework': 'dspy', 'latency_s': 1.781, 'prompt_tokens': 160, 'completion_tokens': 90, 'total_tokens': 250, 'llm_calls': 1}

=== gemini-3.1-flash-lite-preview ===
  {'framework': 'raw', 'latency_s': 0.801, 'prompt_tokens': 26, 'completion_tokens': 18, 'total_tokens': 44}
  {'framework': 'langgraph', 'latency_s': 0.748, 'prompt_tokens': 26, 'completion_tokens': 18, 'total_tokens': 44}
  {'framework': 'crewai', 'latency_s': 0.774, 'prompt_tokens': 182, 'completion_tokens': 39, 'total_tokens': 221, 'llm_calls': 1}
  {'framework': 'dspy', 'latency_s': 0.777, 'prompt_tokens': 160, 'completion_tokens': 31, 'tot

**Cached: byte-level prompt each framework sent (messages_inspected.json):**

In [11]:
p = ROOT / 'results' / 'messages_inspected.json'
if p.exists():
    d = json.loads(p.read_text())
    for fw, msgs in d.items():
        print(f'\n=== {fw} ===')
        for m in msgs:
            print(f'[{m["role"]}]')
            print(m['content'][:600])



=== langgraph ===
[user]
Is a mug refundable if it arrived damaged today under a 30-day damage policy? Reply with one short line.

=== crewai ===
[system]
You are Refund Policy Analyst. You know damage clauses and return windows.
Your personal goal is: Decide eligibility from ticket and policy
To give my best complete final answer to the task respond using the exact following format:

Thought: I now can give a great answer
Final Answer: Your final answer must be the great and the most complete as possible, it must be outcome described.

I MUST use these formats, my job depends on it!
[user]

Current Task: Is a mug refundable if it arrived damaged today under a 30-day damage policy? Reply with one short line.

This is the expected criteria for your final answer: One line.
you MUST return the actual complete content as the final answer, not a summary.

Begin! This is VERY important to you, use the tools available and give your best Final Answer, your job depends on it!

Thought:

=== ds

**Cached: reasoning fraction with bootstrap CIs (probe_reasoning.json):**

In [12]:
import statistics, random, math
p = ROOT / 'results' / 'probe_reasoning.json'
if p.exists():
    d = json.loads(p.read_text())
    def boot(xs, B=2000, seed=42):
        rng = random.Random(seed); n = len(xs); means = []
        for _ in range(B):
            s = [xs[rng.randrange(n)] for _ in range(n)]
            means.append(sum(s) / n)
        means.sort()
        return statistics.mean(means), means[int(0.025*B)], means[int(0.975*B)]
    print(f'{"framework":<10} {"mean reason%":>12} {"95% CI":>20} {"n":>4}')
    for fw, rows in d.items():
        fracs = [r['reasoning']/r['completion'] for r in rows if 'error' not in r and r['completion']]
        m, lo, hi = boot(fracs)
        print(f'{fw:<10} {m*100:>11.1f}% {f"[{lo*100:.1f}%, {hi*100:.1f}%]":>20} {len(fracs):>4}')


framework  mean reason%               95% CI    n
langgraph         90.7%       [89.5%, 91.9%]   30
crewai            89.2%       [87.9%, 90.4%]   30
dspy              64.1%       [60.1%, 68.4%]   23


**Cached: cProfile function-call counts (probe_cprofile.json):**

In [13]:
p = ROOT / 'results' / 'probe_cprofile.json'
if p.exists():
    d = json.loads(p.read_text())
    print(f'{"framework":<10} {"wall_s":>8} {"function_calls (from pstats)":>40}')
    for fw, info in d.items():
        head = info['full_pstats'].splitlines()[0] if info.get('full_pstats') else ''
        print(f'{fw:<10} {info.get("wall_s", 0):>8} {head}')


framework    wall_s             function_calls (from pstats)
langgraph     4.645          80487 function calls (78696 primitive calls) in 4.645 seconds
crewai        5.193          85959 function calls (83008 primitive calls) in 5.193 seconds
dspy          5.268          383933 function calls (383488 primitive calls) in 5.353 seconds


## 7. The DSPy compile question

Two experiments. `compile_dspy.py` runs BootstrapFewShot at five seeds (42-45 plus a no-shuffle baseline). `hand_fewshot_lg.py` runs three hand-picked 4-shot LangGraph variants on the same test set. Comparison answers: is the compile lift attributable to compilation or just to having 4 examples in context?


In [14]:
if RUN_LIVE:
    os.environ['GEMINI_MODEL'] = 'gemini-2.5-flash-lite'
    for s in [42, 43, 44, 45]:
        !cd bench && GEMINI_MODEL=gemini-2.5-flash-lite python compile_dspy.py --seed {s}
    for v in ['v1', 'v2', 'v3']:
        !cd bench && GEMINI_MODEL=gemini-2.5-flash-lite python hand_fewshot_lg.py --variant {v}


**Cached: compile + hand-pick comparison.**

In [15]:
rows = []
# Compiled DSPy seeds
for s in [42, 43, 44, 45]:
    p = ROOT / 'results' / f'dspy_compiled_eval_gemini-2.5-flash-lite_seed{s}.json'
    if p.exists():
        d = json.loads(p.read_text())
        ok = [r for r in d['compiled']['rows'] if 'error' not in r]
        n = len(ok)
        acc = sum(r['correct'] for r in ok) / n
        toks = sum(r['total'] for r in ok) / n
        rows.append(('compiled', f'seed{s}', acc, toks, n))
# Hand-picks
for v in ['v1', 'v2', 'v3']:
    p = ROOT / 'results' / f'hand_fewshot_lg_eval_gemini-2.5-flash-lite_{v}.json'
    if p.exists():
        d = json.loads(p.read_text())
        ok = [r for r in d['rows'] if 'error' not in r]
        n = len(ok)
        acc = sum(r['correct'] for r in ok) / n
        toks = sum(r['prompt']+r['completion'] for r in ok) / n
        rows.append(('hand-4shot', v, acc, toks, n))

print(f'{"variant":<12} {"id":>8} {"acc":>6} {"tokens/ticket":>15} {"n":>4}')
for k, vid, acc, toks, n in rows:
    print(f'{k:<12} {vid:>8} {acc:>5.1%} {toks:>14.0f} {n:>4}')


variant            id    acc   tokens/ticket    n
compiled       seed42 84.0%            822   50
compiled       seed43 70.0%            834   50
compiled       seed44 70.0%            658   50
compiled       seed45 70.0%            737   50
hand-4shot         v1 76.0%            339   50
hand-4shot         v2 76.0%            310   50
hand-4shot         v3 70.0%            322   50


## 8. Where to go next

- Read `article.md` for the full narrative, with the verdict at the end.
- Read `results/analysis_v2.md` for audited stat tables.
- Re-run any cell with `RUN_LIVE=1 GEMINI_API_KEY=... jupyter nbconvert --execute ...` against your own model setup.
- Re-run on your workload by replacing `bench/tickets_100.json` with your own ticket-shaped data; the rest of the bench is workload-agnostic.

The bench code is the artifact. The numbers shipped here are one snapshot.
